# Projet 13 Plusieurs planètes pour mesurer notre incertitude

**Question.** Nos conclusions tiennent-elles quand les paramètres changent ? **Ancrage :** comparaison de modèles, extension Monte-Carlo et IA. Groupe de quatre conseillé.

**Entrées.** initial, params, modules, steps, dt_yr ; graine ; nombre N de simulations ; intervalles de paramètres. Les copies de paramètres et d’état sont indépendantes pour chaque simulation.

**Modèle.** Tirer par exemple tau_T uniformément entre 5 et 15 ans et pref entre 30 et 60 Pa, puis appeler planet_evolution. Garder les stocks initiaux identiques. Utiliser Random(2026) seulement dans cette enveloppe, jamais dans step. Les bornes sont des hypothèses d’incertitude pédagogiques, pas des mesures.

**Sorties.** Un dictionnaire avec seed, samples, histories, summary et failures. summary contient médiane, quantiles empiriques 5 et 95 % et fraction de simulations avec ice_final > 0,9. Les quantiles portent ici sur T_final ; définir cette variable avant de calculer. failures contient les paramètres et l’erreur de chaque simulation invalidée. Ne pas compter un échec numérique comme une planète gelée.

**Données.** Débuter avec 20 simulations de 100 ans à dt = 1 an ; passer à 200 lorsque le coût est acceptable. Garder tau_ice ≥ 20 ans. Comparer à une simulation centrale et répéter l’ensemble avec dt = 0,5 an à durée identique.

**Validation.** Même graine et même configuration : mêmes résultats. Intervalles de largeur nulle : trajectoires identiques. En relançant une simulation seule avec ses paramètres tirés, on doit retrouver sa trajectoire. Pour N = 20, un quantile extrême est très grossier ; afficher N et les échecs.

**Rendu.** ensemble.py, histogramme des T_final et tableau des paramètres. **Ouverture IA :** fabriquer des couples paramètres → résultat avec le moteur puis comparer un prédicteur simple à une référence moyenne sur des simulations réservées. Aucun entraînement n’est obligatoire. Une IA qui imite le moteur hérite de ses simplifications ; elle ne valide pas sa physique.

Lire aussi le contrat commun et le complément eau du dossier collectif.


## Travailler dans le navigateur

Ouvrir ce fichier .ipynb dans https://notebook.basthon.fr ou Colab. Aucun fichier Python annexe ni installation locale ne sont requis. Exécuter les cellules de haut en bas avec Maj + Entrée. Compléter uniquement les zones **À VOUS**, puis relancer les cellules qui suivent. Télécharger le notebook à la fin et le déposer dans votre dossier OneDrive G13. Le notebook ne se sauvegarde pas automatiquement dans OneDrive.

Les fonctions de référence du banc d’essai sont fournies ; elles ne sont pas la routine de votre groupe. Les formules sont dans la fiche ci-dessus et le complément commun.

## À VOUS — raisonnement avant le code

Membres et rôles : …

Entrées, sorties et unités : …

Formule symbolique et hypothèses : …

Calcul manuel : données … ; résultat attendu …

Cas limite prévu : …

In [ ]:
from copy import deepcopy
from math import exp, pi, isfinite, isclose
from random import Random
import matplotlib.pyplot as plt
def base():
    s = {"time_yr": 0., "T_K": 290., "ice": 0.,
         "co2_mol": 7.49e16, "n2_mol": 1.39e20, "o2_mol": 3.74e19,
         "h2o_mol": 0., "mantle_c_mol": 1e20, "rock_c_mol": 0.,
         "fe_mol": 0., "oxide_fe_mol": 0., "population": 0.,
         "fossil_J": 0., "emitted_c_mol": 0., "diag": {}}
    p = {"g_m_s2": 9.81, "area_m2": 5.10e14, "G": 6.67e-11,
         "kB": 1.38e-23, "NA": 6.02e23, "h": 6.63e-34, "c": 3e8,
         "mass_kg": 5.97e24, "radius_m": 6.371e6, "T_exo_K": 1000.,
         "S_W_m2": 1361., "sigma": 5.67e-8, "pref_Pa": 42.,
         "tau_T_yr": 10., "tau_ice_yr": 20.,
         "water_mode": "mixed_screen", "ocean_prescribed": True,
         "volcanic_mol_yr": 0., "weather_mol_yr": 0.,
         "land_fraction": 1., "co2_ref_mol": 7.49e16,
         "oxygen_mol_yr": 0., "lambda_nm": 255.,
         "ozone_column_mol_m2": .10, "absorption_m2_mol": 100.,
         "need_J_person_yr": 1e9, "renewable_J_yr": 4e11,
         "fossil_capacity_J_yr": 3e11, "carbon_mol_J": 2e-6,
         "birth_rate_yr": 0., "death_rate_yr": 0., "deficit_death_rate_yr": 0.}
    return s, p
"""Un moteur deterministe : lire, additionner les flux, avancer, verifier."""
from copy import deepcopy
from math import isfinite

STOCKS = ["co2_mol", "n2_mol", "o2_mol", "h2o_mol", "mantle_c_mol",
          "rock_c_mol", "fe_mol", "oxide_fe_mol", "population", "fossil_J",
          "emitted_c_mol"]
VARIABLES = STOCKS + ["T_K", "ice"]

def verifier_etat(s):
    for cle in VARIABLES + ["time_yr"]:
        if not isfinite(s[cle]) or s[cle] < 0:
            raise ValueError(f"Valeur invalide : {cle} = {s[cle]}")
    if s["ice"] > 1 or s["T_K"] <= 0:
        raise ValueError("Fraction de glace ou temperature invalide")

def verifier_diagnostic(x):
    if isinstance(x, dict):
        for valeur in x.values():
            verifier_diagnostic(valeur)
    elif isinstance(x, (int, float)) and not isfinite(x):
        raise ValueError("Diagnostic non fini")
    elif not isinstance(x, (str, bool, int, float, type(None))):
        raise ValueError("Type de diagnostic invalide")

def evaluer(s, p, modules, dt):
    travail = deepcopy(s)
    travail["diag"] = {}
    taux = {cle: 0.0 for cle in VARIABLES}
    for module in modules:
        try:
            r = module(deepcopy(travail), deepcopy(p), dt)
            if set(r) != {"rates", "values", "diag"} or r["values"]:
                raise ValueError("Contrat de sortie incorrect")
            for cle, valeur in r["rates"].items():
                if cle not in taux or not isfinite(valeur):
                    raise ValueError(f"Taux invalide : {cle}")
                taux[cle] += valeur
            for cle, valeur in r["diag"].items():
                if cle in travail["diag"]:
                    raise ValueError(f"Diagnostic duplique : {cle}")
                verifier_diagnostic(valeur)
                travail["diag"][cle] = valeur
        except (ValueError, KeyError, TypeError) as erreur:
            raise ValueError(f"{module.__module__}, t={s['time_yr']}, dt={dt} : {erreur}") from erreur
    return travail, taux

def planet_evolution(initial, params, modules, steps, dt_yr):
    if not isfinite(dt_yr) or dt_yr <= 0:
        raise ValueError("Le pas doit etre positif et fini")
    if not isinstance(steps, int) or steps < 0:
        raise ValueError("Le nombre de pas doit etre entier et >= 0")
    s = deepcopy(initial)
    verifier_etat(s)
    histoire = []
    for numero in range(steps + 1):
        # On enregistre les diagnostics du MEME instant que les stocks.
        travail, taux = evaluer(s, params, modules, dt_yr)
        histoire.append(deepcopy(travail))
        if numero == steps:
            break
        suivant = deepcopy(travail)
        for cle in VARIABLES:
            suivant[cle] = s[cle] + dt_yr * taux[cle]
        suivant["time_yr"] += dt_yr
        verifier_etat(suivant)
        s = suivant
    return histoire

def simuler_isole(initial, params, routine, steps, dt):
    """Banc d'essai : diagnostics voisins prescrits et conservés.
    Ce banc ne remplace pas le moteur collectif, qui recalcule les voisins.
    """
    s=deepcopy(initial);h=[deepcopy(s)]
    for _ in range(steps):
        r=routine(deepcopy(s),deepcopy(params),dt)
        assert set(r)=={'rates','values','diag'} and not r['values']
        for cle,taux in r['rates'].items():
            s[cle]+=dt*taux
            assert isfinite(s[cle]) and s[cle]>=0,cle
        s['time_yr']+=dt
        if 'ice' in s:assert s['ice']<=1
        h.append(deepcopy(s))
    return h


## Référence fournie pour Monte-Carlo

Le banc fait évoluer une température vers une cible avec pression CO2 et température radiative prescrites. Il sert à développer votre enveloppe sans attendre les autres groupes. Au raccordement, MODULES sera remplacé par les routines de la classe.

In [ ]:
# Projet 10 : loi de serre fictive, Euler explicite.
def climat_reference(s, p, dt):
    if dt > p["tau_T_yr"]:
        raise ValueError("dt doit etre <= tau_T_yr")
    co2 = s["diag"]["pco2_Pa"]
    serre = 66 * co2 / (co2 + p["pref_Pa"])
    cible = s["diag"]["Teq_K"] + serre
    taux = (cible - s["T_K"]) / p["tau_T_yr"]
    return {"rates": {"T_K": taux}, "values": {},
            "diag": {"Ttarget_K": cible, "greenhouse_K": serre}}

def diagnostics_reference(s,p,dt):
    return {"rates":{},"values":{},"diag":{"Teq_K":254.6,"pco2_Pa":42.}}
MODULES=[diagnostics_reference,climat_reference]


In [ ]:
def quantile(valeurs, fraction):
    ordre = sorted(valeurs)
    position = fraction * (len(ordre) - 1)
    i = int(position)
    j = min(i + 1, len(ordre) - 1)
    return ordre[i] + (ordre[j] - ordre[i]) * (position - i)



## À VOUS — écrire la routine

Initialiser Random(seed), copier les paramètres, tirer les valeurs dans les bornes, appeler le moteur, conserver les échecs, calculer la médiane et les quantiles. Relire les noms exacts de sortie dans la fiche.

Remplacer le `raise NotImplementedError` par votre code. Les tests indiquent les valeurs attendues ; ils ne donnent pas votre implémentation.

In [ ]:
def ensemble(initial, params, modules, steps, dt, nombre=50, seed=2026, bornes=None):
    # À VOUS : écrire ici le modèle donné dans la fiche.
    raise NotImplementedError("routine du groupe 13")


## Vérifier votre fonction

Avant de coder, la cellule affiche « À compléter ». Une fois la routine écrite, une assertion qui échoue signifie que le test doit être examiné. Noter **attendu / obtenu / conclusion**, même quand les assertions passent.

In [ ]:
try:
    s,p=base();a=ensemble(s,p,MODULES,10,1,nombre=3);b=ensemble(s,p,MODULES,10,1,nombre=3)
    assert a==b
    r=ensemble(s,p,MODULES,10,1,nombre=3,bornes={'pref_Pa':(42.,42.)})
    assert r['histories'][0]==r['histories'][1]
    r=ensemble(s,p,MODULES,10,1,nombre=2,bornes={'tau_T_yr':(.1,.1)})
    assert len(r['failures'])==2 and r['summary']=={}
    print("Vérifications numériques réussies. Ajouter votre propre essai ci-dessous.")
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — ajouter une vérification justifiée

Quel paramètre changez-vous ? Quel résultat prévoyez-vous et pourquoi ?

In [ ]:
# À VOUS : ajouter un test avec un résultat attendu et une justification.


## Première figure fournie comme point de départ

La cellule propose une expérience et des axes. Exécutez-la après avoir codé votre routine. Ensuite, modifiez un paramètre, comparez deux expériences et justifiez votre lecture. Ce graphique isolé n’est pas encore la planète collective.

In [ ]:
try:
    plt.figure(figsize=(7,4))
    s,p=base();r=ensemble(s,p,MODULES,100,1,nombre=30)
    y=[h[-1]['T_K'] for h in r['histories'] if h is not None]
    assert y,'Aucune simulation valide : lire failures'
    plt.hist(y,bins=8);plt.xlabel('Température finale (K)');plt.ylabel('Nombre de mondes')
    print('Simulations demandées :',len(r['samples']),'échecs :',len(r['failures']));print(r['summary'])
    plt.title("Groupe 13 — ensemble")
    plt.grid(alpha=.25)
    plt.show()
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — interprétation et raccordement

Prédiction avant la figure : …

Ce que montre la courbe : …

Explication par la formule : …

Limite de notre modèle : …

Voisin fournisseur et donnée de substitution à retirer : …

Sortie utilisée par un autre groupe : …

Contribution au graphe collectif et signe de la rétroaction : …

Livrer ce notebook complété, avec les résultats visibles et le nom des membres.

## Tester notre routine dans le projet commun

Le lien de classe Basthon charge aussi `classe_commune.py` depuis GitHub. Les autres groupes proviennent de la version commune affichée. Votre fonction locale remplace uniquement votre groupe, sans changer le dépôt. Si vous avez ouvert seulement le fichier .ipynb, charger aussi classe_commune.py comme module avec le bouton d’ouverture.

Une sauvegarde Basthon télécharge votre copie ; elle ne publie pas votre travail sur GitHub. Déposer le notebook et les résultats sur OneDrive. Après validation, le professeur met à jour la version commune ; les autres groupes la récupèrent au prochain chargement. Les références professeur restent identifiées tant qu’une routine élève n’a pas été validée.


In [ ]:
try:
    import classe_commune as commun
    print('Version commune :',commun.VERSION_CLASSE)
    s,p,n,dt=commun.scenario_classe('societe')
    # Votre enveloppe Monte-Carlo appelle toutes les routines communes.
    resultat_classe=ensemble(s,p,commun.MODULES,n,dt,nombre=20,seed=2026)
    print(resultat_classe['summary'])
    from IPython.display import HTML, display
    from base64 import b64encode
    import json
    encoded=b64encode(json.dumps(resultat_classe,ensure_ascii=False).encode()).decode()
    display(HTML('<a download="G13_resultats.json" href="data:application/octet-stream;base64,'+encoded+'">Télécharger nos résultats collectifs</a>'))
except ImportError:
    print('Ouvrir le lien Basthon de classe, ou charger classe_commune.py comme module.')
except NotImplementedError as erreur:
    print('Compléter d’abord votre enveloppe :',erreur)


## Notre bilan collectif

Quelle version commune avons-nous utilisée ? …

Quelles routines viennent de nos camarades et lesquelles restent des références ? …

Quel effet produit notre routine dans ce scénario ? …

Déposer le notebook, le rapport et les résultats dans le dossier du groupe.